# Phase 1 — NLCD zonal stats (fractional land cover)

**Phase 1 screening:** Widen the zonal buffer to recover edge cells with `nlcd_pixel_count == 0`.

| Input | Output |
|-------|--------|
| `data/processed/weather_cell_features.parquet` | `data/processed/grid_nlcd_fractional.parquet` |
| `data/Annual_NLCD_LndCov_2024/Annual_NLCD_LndCov_2024_CU_C1V1.tif` | |

**Method:** ~10 km cells → buffer in NLCD projection → `rasterstats` categorical counts → fractions per NLCD class.

**Zonal buffer:** `CELL_RADIUS_M` is **7000 m** (widened from 5000 m to recover edge cells at the coast and grid boundary).

> **Runtime:** ~4,400 cells, expect **5–20 min** depending on disk I/O. Progress prints per chunk.

In [1]:
# --- User settings ---
CELL_RADIUS_M = 7000       # 7 km buffer (widened from 5 km to recover edge cells)
CHUNK_SIZE = 200           # cells per zonal-stats batch
USE_SCRATCH_CACHE = True   # skip recompute if scratch parquet exists
INVALIDATE_SCRATCH = True  # force recompute after radius change

import shutil
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import geopandas as gpd
import polars as pl
import rasterio
import yaml
from pyproj import Transformer
from rasterstats import zonal_stats
from shapely.geometry import Point

ROOT = Path('../..').resolve()
CONFIG_PATH = ROOT / 'config' / 'paths.yaml'
WEATHER_PATH = ROOT / 'data' / 'processed' / 'weather_cell_features.parquet'
NLCD_TIF = ROOT / 'data' / 'Annual_NLCD_LndCov_2024' / 'Annual_NLCD_LndCov_2024_CU_C1V1.tif'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'grid_nlcd_fractional.parquet'
SCRATCH_PATH = ROOT / 'data' / 'processed' / 'scratch' / 'grid_nlcd_zonal_raw.parquet'

with CONFIG_PATH.open(encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
SCRATCH_PATH.parent.mkdir(parents=True, exist_ok=True)

if INVALIDATE_SCRATCH and SCRATCH_PATH.exists():
    SCRATCH_PATH.unlink()
    print('Invalidated NLCD scratch cache:', SCRATCH_PATH)

if not WEATHER_PATH.exists():
    raise FileNotFoundError(f'Run 01_ingest_weather.ipynb first: {WEATHER_PATH}')
if not NLCD_TIF.exists():
    raise FileNotFoundError(f'NLCD GeoTIFF not found: {NLCD_TIF}')

cells = (
    pl.read_parquet(WEATHER_PATH)
    .select(['latitude', 'longitude'])
    .drop_nulls()
    .unique()
    .sort(['latitude', 'longitude'])
)

print('Repo root   :', ROOT)
print('Weather cells:', len(cells))
print('Cell radius :', CELL_RADIUS_M, 'm')
print('NLCD tif    :', NLCD_TIF)
print('Output      :', OUTPUT_PATH)

Invalidated NLCD scratch cache: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/scratch/grid_nlcd_zonal_raw.parquet


Repo root   : /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York
Weather cells: 4401
Cell radius : 7000 m
NLCD tif    : /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/Annual_NLCD_LndCov_2024/Annual_NLCD_LndCov_2024_CU_C1V1.tif
Output      : /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/grid_nlcd_fractional.parquet


In [2]:
NLCD_NODATA = 250

NLCD_CLASSES = {
    11: 'Open Water',
    12: 'Perennial Ice/Snow',
    21: 'Developed, Open Space',
    22: 'Developed, Low Intensity',
    23: 'Developed, Medium Intensity',
    24: 'Developed, High Intensity',
    31: 'Barren Land',
    41: 'Deciduous Forest',
    42: 'Evergreen Forest',
    43: 'Mixed Forest',
    52: 'Shrub/Scrub',
    71: 'Herbaceous',
    81: 'Hay/Pasture',
    82: 'Cultivated Crops',
    90: 'Woody Wetlands',
    95: 'Emergent Herbaceous Wetlands',
}
NLCD_CODES = sorted(NLCD_CLASSES)
FRAC_COLS = [f'frac_class_{code}' for code in NLCD_CODES]


def sample_centroid_nlcd(lat: float, lon: float, transformer: Transformer, dataset) -> int | None:
    x, y = transformer.transform(lon, lat)
    val = int(list(dataset.sample([(x, y)]))[0][0])
    return None if val == NLCD_NODATA else val


def zonal_fractions_for_chunk(chunk: pl.DataFrame, nlcd_tif: Path, cell_radius_m: float) -> pl.DataFrame:
    pdf = chunk.to_pandas()
    gdf = gpd.GeoDataFrame(
        pdf,
        geometry=[Point(r.longitude, r.latitude) for r in pdf.itertuples()],
        crs='EPSG:4326',
    )

    with rasterio.open(nlcd_tif) as src:
        gdf = gdf.to_crs(src.crs)
        gdf['geometry'] = gdf.geometry.buffer(cell_radius_m)
        transformer = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True)
        stats = zonal_stats(
            gdf.geometry,
            str(nlcd_tif),
            categorical=True,
            nodata=NLCD_NODATA,
        )
        centroid_codes = [
            sample_centroid_nlcd(r.latitude, r.longitude, transformer, src)
            for r in pdf.itertuples()
        ]

    rows: list[dict] = []
    for i, r in enumerate(pdf.itertuples()):
        counts = stats[i] or {}
        total = sum(counts.values())
        row = {'latitude': r.latitude, 'longitude': r.longitude}
        for code in NLCD_CODES:
            row[f'frac_class_{code}'] = (counts.get(code, 0) / total) if total else None
        row['nlcd_pixel_count'] = total
        row['nlcd_code_centroid'] = centroid_codes[i]
        row['nlcd_class_centroid'] = (
            NLCD_CLASSES.get(centroid_codes[i], 'Unknown/NoData') if centroid_codes[i] else 'Unknown/NoData'
        )
        row['nlcd_fraction_sum'] = sum(row[f'frac_class_{c}'] or 0.0 for c in NLCD_CODES)
        rows.append(row)

    return pl.DataFrame(rows)

In [3]:
if USE_SCRATCH_CACHE and SCRATCH_PATH.exists():
    print(f'Loading cached zonal stats: {SCRATCH_PATH.name}')
    nlcd_df = pl.read_parquet(SCRATCH_PATH)
else:
    chunks: list[pl.DataFrame] = []
    n = len(cells)
    for start in range(0, n, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, n)
        chunk = cells.slice(start, end - start)
        print(f'Zonal stats cells {start + 1}-{end} / {n} ...')
        chunks.append(zonal_fractions_for_chunk(chunk, NLCD_TIF, CELL_RADIUS_M))
    nlcd_df = pl.concat(chunks, how='vertical')
    nlcd_df.write_parquet(SCRATCH_PATH)
    print(f'Cached scratch -> {SCRATCH_PATH}')

nlcd_df.write_parquet(OUTPUT_PATH)
print(f'\nWrote {len(nlcd_df):,} cells x {len(nlcd_df.columns)} columns')
print('Output:', OUTPUT_PATH)

Zonal stats cells 1-200 / 4401 ...


Zonal stats cells 201-400 / 4401 ...


Zonal stats cells 401-600 / 4401 ...


Zonal stats cells 601-800 / 4401 ...


Zonal stats cells 801-1000 / 4401 ...


...

Zonal stats cells 3601-3800 / 4401 ...


Zonal stats cells 3801-4000 / 4401 ...


Zonal stats cells 4001-4200 / 4401 ...


Zonal stats cells 4201-4400 / 4401 ...


Zonal stats cells 4401-4401 / 4401 ...
Cached scratch -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/scratch/grid_nlcd_zonal_raw.parquet

Wrote 4,401 cells x 22 columns
Output: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/grid_nlcd_fractional.parquet


In [4]:
# QA — fraction sums should be ~1.0 where pixels were found
qa = nlcd_df.select([
    pl.len().alias('n_cells'),
    pl.col('nlcd_pixel_count').mean().alias('mean_pixel_count'),
    pl.col('nlcd_fraction_sum').mean().alias('mean_fraction_sum'),
    pl.col('nlcd_fraction_sum').min().alias('min_fraction_sum'),
    pl.col('nlcd_fraction_sum').max().alias('max_fraction_sum'),
    (pl.col('nlcd_pixel_count') == 0).sum().alias('cells_no_nlcd_pixels'),
])
print(f'CELL_RADIUS_M = {CELL_RADIUS_M}')
print('v1 baseline cells_no_nlcd_pixels = 721 (CELL_RADIUS_M = 5000)')
qa

CELL_RADIUS_M = 7000
v1 baseline cells_no_nlcd_pixels = 721 (CELL_RADIUS_M = 5000)


n_cells,mean_pixel_count,mean_fraction_sum,min_fraction_sum,max_fraction_sum,cells_no_nlcd_pixels
u32,f64,f64,f64,f64,u32
4401,140645.072484,0.840264,0.0,1.0,703


In [5]:
# Top land-cover classes statewide (by mean fraction)
mean_fracs = {col: nlcd_df[col].mean() for col in FRAC_COLS}
top = sorted(mean_fracs.items(), key=lambda x: x[1] or 0, reverse=True)[:8]
pl.DataFrame({
    'nlcd_code': [int(k.split('_')[-1]) for k, _ in top],
    'class': [NLCD_CLASSES[int(k.split('_')[-1])] for k, _ in top],
    'mean_fraction': [v for _, v in top],
})

nlcd_code,class,mean_fraction
i64,str,f64
41,"""Deciduous Forest""",0.37514
43,"""Mixed Forest""",0.125277
81,"""Hay/Pasture""",0.116359
11,"""Open Water""",0.101148
21,"""Developed, Open Space""",0.060852
90,"""Woody Wetlands""",0.057773
82,"""Cultivated Crops""",0.046897
42,"""Evergreen Forest""",0.038833


In [6]:
nlcd_df.head(5)

latitude,longitude,frac_class_11,frac_class_12,frac_class_21,frac_class_22,frac_class_23,frac_class_24,frac_class_31,frac_class_41,frac_class_42,frac_class_43,frac_class_52,frac_class_71,frac_class_81,frac_class_82,frac_class_90,frac_class_95,nlcd_pixel_count,nlcd_code_centroid,nlcd_class_centroid,nlcd_fraction_sum
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,str,f64
40.5,-79.8,0.03325,0.0,0.244226,0.218607,0.125744,0.025093,0.003373,0.300583,0.000047,0.010611,0.000826,0.003139,0.033912,0.000574,0.000012,0.000006,170768,41,"""Deciduous Forest""",1.0
40.5,-79.71,0.004538,0.0,0.188044,0.12383,0.050874,0.006371,0.00072,0.512871,0.000609,0.031732,0.001528,0.003982,0.074291,0.000609,0.0,0.0,170774,41,"""Deciduous Forest""",1.0
40.5,-79.62,0.025802,0.0,0.123706,0.051974,0.017252,0.002793,0.001189,0.570321,0.002296,0.052155,0.001927,0.005159,0.143406,0.00195,0.000018,0.000053,170760,41,"""Deciduous Forest""",1.0
40.5,-79.53,0.038882,0.0,0.083216,0.037242,0.008479,0.001604,0.004017,0.579284,0.003519,0.040785,0.004948,0.009287,0.154052,0.034361,0.000217,0.000105,170773,41,"""Deciduous Forest""",1.0
40.5,-79.44,0.020971,0.0,0.076641,0.032526,0.006366,0.000949,0.00181,0.588163,0.002676,0.025908,0.006161,0.005499,0.145704,0.086368,0.000123,0.000135,170757,21,"""Developed, Open Space""",1.0
